In [1]:
import pandas as pd
import pyarrow.parquet as pq

file_path = "../data/raw/fhvhv_tripdata_2025-01.parquet"
parquet_file = pq.ParquetFile(file_path)

TOTAL_ROWS = parquet_file.metadata.num_rows

print(f"Total raw records: {TOTAL_ROWS:,}")

Total raw records: 20,405,666


In [3]:
columns = [
    "request_datetime",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "tips",
    "driver_pay"
]

issue_samples = {
    "zero_distance": [],
    "zero_trip_time": [],
    "negative_fare": [],
    "pickup_before_request": []
}

for batch in parquet_file.iter_batches(
    batch_size=500_000,
    columns=columns
):
    df = batch.to_pandas()

    conditions = {
        "zero_distance": df["trip_miles"] == 0,
        "zero_trip_time": df["trip_time"] == 0,
        "negative_fare": df["base_passenger_fare"] < 0,
        "pickup_before_request":
            df["pickup_datetime"] < df["request_datetime"]
    }

    for name, condition in conditions.items():

        if len(issue_samples[name]) < 20:

            remaining = 20 - len(issue_samples[name])

            records = df.loc[condition].head(remaining)

            issue_samples[name].extend(
                records.to_dict("records")
            )

In [4]:
pd.DataFrame(issue_samples["zero_distance"])

,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tips,driver_pay
0,2025-01-01 00:50:19,2025-01-01 00:52:40,2025-01-01 00:54:20,193,193,0.0,100,20.29,0.0,3.99
1,2025-01-01 00:09:12,2025-01-01 00:11:22,2025-01-01 00:12:43,50,50,0.0,81,23.41,0.0,1.50
2,2025-01-01 00:06:52,2025-01-01 00:17:30,2025-01-01 00:55:34,163,163,0.0,2284,34.79,0.0,26.60
3,2025-01-01 00:39:19,2025-01-01 00:41:43,2025-01-01 00:44:18,158,158,0.0,155,12.22,0.0,13.74
4,2024-12-31 23:59:31,2025-01-01 00:01:02,2025-01-01 00:11:10,211,211,0.0,608,8.67,0.0,7.13
5,2025-01-01 00:46:41,2025-01-01 00:56:18,2025-01-01 01:04:11,130,130,0.0,473,10.07,0.0,15.21
6,2025-01-01 00:38:45,2025-01-01 00:47:45,2025-01-01 01:11:20,10,10,0.0,1415,16.69,0.0,9.71
7,2025-01-01 00:10:32,2025-01-01 00:13:39,2025-01-01 00:17:02,79,79,0.0,203,9.13,0.0,10.50
8,2025-01-01 00:15:41,2025-01-01 00:16:50,2025-01-01 00:18:57,16,16,0.0,127,13.91,0.0,18.62
9,2025-01-01 00:32:25,2025-01-01 00:39:13,2025-01-01 00:43:25,47,47,0.0,252,73.22,0.0,24.05


In [5]:
pd.DataFrame(issue_samples["zero_trip_time"])

,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tips,driver_pay
0,2025-01-03 15:21:03,2025-01-03 15:32:04,2025-01-03 15:32:04,220,220,0.0,0,8.87,5.0,4.77
1,2025-01-07 10:40:00,2025-01-07 10:38:03,2025-01-07 10:38:03,20,20,0.0,0,8.87,0.0,4.48
2,2025-01-17 16:09:35,2025-01-17 16:12:30,2025-01-17 16:12:30,75,75,0.0,0,7.19,0.0,0.00
3,2025-01-26 00:32:06,2025-01-26 00:39:47,2025-01-26 00:39:47,91,91,0.0,0,5.39,0.0,4.00


In [6]:
pd.DataFrame(issue_samples["negative_fare"])

,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tips,driver_pay
0,2025-01-01 00:11:38,2025-01-01 00:13:52,2025-01-01 00:27:11,170,145,3.140,799,-6.39,2.30,17.00
1,2025-01-01 00:15:03,2025-01-01 00:23:06,2025-01-01 00:30:02,174,20,1.820,416,-1.12,0.00,7.16
2,2025-01-01 00:34:50,2025-01-01 00:38:39,2025-01-01 01:03:55,164,162,1.700,1516,-0.75,0.00,23.24
3,2025-01-01 00:06:27,2025-01-01 00:08:55,2025-01-01 00:54:15,75,107,8.930,2720,-3.09,1.00,28.90
4,2025-01-01 00:18:01,2025-01-01 00:25:08,2025-01-01 00:34:58,107,79,1.450,590,-2.75,0.00,12.78
5,2025-01-01 00:23:48,2025-01-01 00:29:31,2025-01-01 00:33:33,179,7,0.780,242,-1.64,0.09,7.53
6,2025-01-01 00:22:56,2025-01-01 00:29:28,2025-01-01 00:46:04,13,97,4.150,996,-2.75,5.00,16.51
7,2025-01-01 01:15:14,2025-01-01 01:26:51,2025-01-01 01:35:03,137,145,2.170,492,-3.47,3.00,27.70
8,2025-01-01 00:43:45,2025-01-01 01:05:44,2025-01-01 01:19:52,170,79,1.310,848,-2.75,0.00,16.08
9,2025-01-01 02:00:01,2025-01-01 02:08:57,2025-01-01 02:15:07,243,116,1.480,370,-2.00,1.00,13.06


In [7]:
pd.DataFrame(issue_samples["pickup_before_request"])

,request_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tips,driver_pay
0,2025-01-01 00:30:01,2025-01-01 00:29:40,2025-01-01 01:17:54,148,151,8.134,2894,63.99,0.00,42.56
1,2025-01-01 00:30:01,2025-01-01 00:30:00,2025-01-01 00:42:41,90,170,1.727,761,29.47,0.00,17.77
2,2025-01-01 00:25:00,2025-01-01 00:23:14,2025-01-01 01:00:28,41,164,7.100,2234,17.56,0.00,29.24
3,2025-01-01 01:00:00,2025-01-01 00:59:51,2025-01-01 01:20:14,249,265,3.870,1223,22.61,4.48,30.60
4,2025-01-01 00:30:01,2025-01-01 00:30:00,2025-01-01 00:45:37,13,224,5.065,937,43.02,0.00,16.05
5,2025-01-01 01:00:01,2025-01-01 00:57:38,2025-01-01 01:20:27,239,170,3.820,1369,47.61,11.18,28.56
6,2025-01-01 00:30:00,2025-01-01 00:23:02,2025-01-01 00:44:40,146,234,4.640,1298,9.82,0.00,22.01
7,2025-01-01 00:40:00,2025-01-01 00:33:02,2025-01-01 01:04:19,36,188,4.940,1877,18.43,0.00,11.84
8,2025-01-01 00:45:01,2025-01-01 00:45:00,2025-01-01 01:06:48,137,68,1.072,1308,35.86,0.00,16.71
9,2025-01-01 00:45:00,2025-01-01 00:44:21,2025-01-01 01:16:51,256,225,4.750,1950,14.34,1.00,27.35


In [8]:
import os
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

input_path = "../data/raw/fhvhv_tripdata_2025-01.parquet"
output_path = "../data/processed/trips_analytics_2025_01.parquet"

columns = [
    "hvfhs_license_num",
    "request_datetime",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_miles",
    "trip_time",
    "base_passenger_fare",
    "tips",
    "driver_pay",
    "shared_request_flag",
    "shared_match_flag"
]

source = pq.ParquetFile(input_path)

writer = None
rows_written = 0

for batch in source.iter_batches(
    batch_size=500_000,
    columns=columns
):
    df = batch.to_pandas()

    # Keep trips whose pickup occurred during January 2025
    df = df[
        (df["pickup_datetime"] >= "2025-01-01") &
        (df["pickup_datetime"] < "2025-02-01")
    ].copy()

    # Useful derived variables
    df["trip_duration_min"] = df["trip_time"] / 60

    df["pickup_date"] = df["pickup_datetime"].dt.date
    df["pickup_hour"] = df["pickup_datetime"].dt.hour
    df["pickup_day"] = df["pickup_datetime"].dt.day_name()
    df["is_weekend"] = (
        df["pickup_datetime"].dt.dayofweek >= 5
    )

    # Quality flags - preserve data instead of blindly deleting it
    df["valid_distance"] = df["trip_miles"] > 0
    df["valid_duration"] = df["trip_time"] > 0
    df["valid_fare"] = df["base_passenger_fare"] >= 0

    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)
    rows_written += len(df)

if writer is not None:
    writer.close()

print(f"Rows written: {rows_written:,}")
print("Saved to:", output_path)

Rows written: 20,405,666
Saved to: ../data/processed/trips_analytics_2025_01.parquet


In [9]:
processed_path = "../data/processed/trips_analytics_2025_01.parquet"

processed_file = pq.ParquetFile(processed_path)

print("Rows:", processed_file.metadata.num_rows)
print("Columns:", processed_file.metadata.num_columns)
print("\nColumns:")
print(processed_file.schema.names)

Rows: 20405666
Columns: 21

Columns:
['hvfhs_license_num', 'request_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'trip_duration_min', 'pickup_date', 'pickup_hour', 'pickup_day', 'is_weekend', 'valid_distance', 'valid_duration', 'valid_fare']


In [10]:
sample = next(
    processed_file.iter_batches(batch_size=10)
).to_pandas()

sample[
    [
        "pickup_datetime",
        "trip_time",
        "trip_duration_min",
        "pickup_date",
        "pickup_hour",
        "pickup_day",
        "is_weekend",
        "valid_distance",
        "valid_duration",
        "valid_fare"
    ]
]

,pickup_datetime,trip_time,trip_duration_min,pickup_date,pickup_hour,pickup_day,is_weekend,valid_distance,valid_duration,valid_fare
0,2025-01-01 00:33:25,1259,20.983333,2025-01-01,0,Wednesday,False,True,True,True
1,2025-01-01 00:29:49,723,12.050000,2025-01-01,0,Wednesday,False,True,True,True
2,2025-01-01 00:32:39,2874,47.900000,2025-01-01,0,Wednesday,False,True,True,True
3,2025-01-01 00:34:55,264,4.400000,2025-01-01,0,Wednesday,False,True,True,True
4,2025-01-01 00:46:19,457,7.616667,2025-01-01,0,Wednesday,False,True,True,True
5,2025-01-01 00:45:50,2073,34.550000,2025-01-01,0,Wednesday,False,True,True,True
6,2025-01-01 00:25:37,1027,17.116667,2025-01-01,0,Wednesday,False,True,True,True
7,2025-01-01 00:34:51,2891,48.183333,2025-01-01,0,Wednesday,False,True,True,True
8,2025-01-01 00:12:35,2342,39.033333,2025-01-01,0,Wednesday,False,True,True,True
9,2025-01-01 00:58:49,3394,56.566667,2025-01-01,0,Wednesday,False,True,True,True


In [11]:
flag_counts = {
    "invalid_distance": 0,
    "invalid_duration": 0,
    "invalid_fare": 0
}

total = 0

for batch in processed_file.iter_batches(
    batch_size=500_000,
    columns=[
        "valid_distance",
        "valid_duration",
        "valid_fare"
    ]
):
    df = batch.to_pandas()

    total += len(df)

    flag_counts["invalid_distance"] += (~df["valid_distance"]).sum()
    flag_counts["invalid_duration"] += (~df["valid_duration"]).sum()
    flag_counts["invalid_fare"] += (~df["valid_fare"]).sum()

print(f"Total records: {total:,}")

for name, count in flag_counts.items():
    percentage = count / total * 100
    print(f"{name}: {count:,} ({percentage:.4f}%)")

Total records: 20,405,666
invalid_distance: 2,606 (0.0128%)
invalid_duration: 4 (0.0000%)
invalid_fare: 2,427 (0.0119%)


In [12]:
from collections import defaultdict

demand_counts = defaultdict(int)

for batch in processed_file.iter_batches(
    batch_size=500_000,
    columns=[
        "pickup_datetime",
        "PULocationID"
    ]
):
    df = batch.to_pandas()

    df["pickup_hour_ts"] = (
        df["pickup_datetime"].dt.floor("h")
    )

    batch_counts = (
        df.groupby(
            ["pickup_hour_ts", "PULocationID"]
        )
        .size()
    )

    for (hour, zone), count in batch_counts.items():
        demand_counts[(hour, zone)] += count

print("Observed zone-hour combinations:", len(demand_counts))

Observed zone-hour combinations: 187020


In [13]:
demand_df = pd.DataFrame(
    [
        {
            "datetime": hour,
            "PULocationID": zone,
            "demand": count
        }
        for (hour, zone), count in demand_counts.items()
    ]
)

demand_df = demand_df.sort_values(
    ["datetime", "PULocationID"]
).reset_index(drop=True)

print("Rows:", len(demand_df))

demand_df.head(10)

Rows: 187020


,datetime,PULocationID,demand
0,2025-01-01,3,74
1,2025-01-01,4,242
2,2025-01-01,5,18
3,2025-01-01,6,15
4,2025-01-01,7,616
5,2025-01-01,8,5
6,2025-01-01,9,31
7,2025-01-01,10,134
8,2025-01-01,11,58
9,2025-01-01,12,10


In [14]:
print("Rows:", len(demand_df))

print(
    "Total trips in processed data:",
    processed_file.metadata.num_rows
)

print(
    "Total trips represented in demand data:",
    demand_df["demand"].sum()
)

print("\nDate/time range:")
print("Start:", demand_df["datetime"].min())
print("End:", demand_df["datetime"].max())

print("\nDemand statistics:")
print(demand_df["demand"].describe())

Rows: 187020
Total trips in processed data: 20405666
Total trips represented in demand data: 20405666

Date/time range:
Start: 2025-01-01 00:00:00
End: 2025-01-31 23:00:00

Demand statistics:
count    187020.000000
mean        109.109539
std         117.459633
min           1.000000
25%          29.000000
50%          73.000000
75%         151.000000
max        1535.000000
Name: demand, dtype: float64


In [17]:
zone_path = "../data/raw/taxi_zone_lookup.csv"

zones = pd.read_csv(zone_path)

print("Number of zones:", len(zones))
zones.head()

Number of zones: 265


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [18]:
model_zones = zones[
    ~zones["LocationID"].isin([264, 265])
].copy()

print("Modeling zones:", len(model_zones))

model_zones.head()

Modeling zones: 263


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [19]:
all_hours = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-01-31 23:00:00",
    freq="h"
)

print("Number of hours:", len(all_hours))
print("First hour:", all_hours.min())
print("Last hour:", all_hours.max())

Number of hours: 744
First hour: 2025-01-01 00:00:00
Last hour: 2025-01-31 23:00:00


In [20]:
full_grid = pd.MultiIndex.from_product(
    [
        all_hours,
        model_zones["LocationID"]
    ],
    names=["datetime", "PULocationID"]
).to_frame(index=False)

print("Complete grid rows:", len(full_grid))

full_grid.head(10)

Complete grid rows: 195672


,datetime,PULocationID
0,2025-01-01,1
1,2025-01-01,2
2,2025-01-01,3
3,2025-01-01,4
4,2025-01-01,5
5,2025-01-01,6
6,2025-01-01,7
7,2025-01-01,8
8,2025-01-01,9
9,2025-01-01,10


In [21]:
demand_complete = full_grid.merge(
    demand_df,
    on=["datetime", "PULocationID"],
    how="left"
)

# No observed pickups in that zone-hour -> demand = 0
demand_complete["demand"] = (
    demand_complete["demand"]
    .fillna(0)
    .astype("int32")
)

print("Rows:", len(demand_complete))
print(
    "Zero-demand rows:",
    (demand_complete["demand"] == 0).sum()
)

demand_complete.head(10)

Rows: 195672
Zero-demand rows: 9112


,datetime,PULocationID,demand
0,2025-01-01,1,0
1,2025-01-01,2,0
2,2025-01-01,3,74
3,2025-01-01,4,242
4,2025-01-01,5,18
5,2025-01-01,6,15
6,2025-01-01,7,616
7,2025-01-01,8,5
8,2025-01-01,9,31
9,2025-01-01,10,134


In [22]:
original_total = demand_df["demand"].sum()

special_zone_trips = demand_df.loc[
    demand_df["PULocationID"].isin([264, 265]),
    "demand"
].sum()

modeling_total = demand_complete["demand"].sum()

print(f"Original total trips: {original_total:,}")
print(f"Trips in zones 264/265: {special_zone_trips:,}")
print(f"Expected modeling total: {original_total - special_zone_trips:,}")
print(f"Actual modeling total: {modeling_total:,}")

print(
    "Validation passed:",
    original_total - special_zone_trips == modeling_total
)

print("\nDemand statistics:")
print(demand_complete["demand"].describe())

Original total trips: 20,405,666
Trips in zones 264/265: 773
Expected modeling total: 20,404,893
Actual modeling total: 20,404,893
Validation passed: True

Demand statistics:
count    195672.000000
mean        104.281108
std         117.007058
min           0.000000
25%          24.000000
50%          69.000000
75%         145.000000
max        1535.000000
Name: demand, dtype: float64


In [23]:
zone_info = model_zones[
    ["LocationID", "Borough", "Zone", "service_zone"]
].rename(
    columns={"LocationID": "PULocationID"}
)

demand_final = demand_complete.merge(
    zone_info,
    on="PULocationID",
    how="left"
)

demand_final.head()

,datetime,PULocationID,demand,Borough,Zone,service_zone
0,2025-01-01,1,0,EWR,Newark Airport,EWR
1,2025-01-01,2,0,Queens,Jamaica Bay,Boro Zone
2,2025-01-01,3,74,Bronx,Allerton/Pelham Gardens,Boro Zone
3,2025-01-01,4,242,Manhattan,Alphabet City,Yellow Zone
4,2025-01-01,5,18,Staten Island,Arden Heights,Boro Zone


In [24]:
demand_final["date"] = demand_final["datetime"].dt.date
demand_final["hour"] = demand_final["datetime"].dt.hour
demand_final["day_of_week"] = demand_final["datetime"].dt.day_name()
demand_final["day_of_week_num"] = demand_final["datetime"].dt.dayofweek
demand_final["day_of_month"] = demand_final["datetime"].dt.day
demand_final["month"] = demand_final["datetime"].dt.month

demand_final["is_weekend"] = (
    demand_final["day_of_week_num"] >= 5
)

In [25]:
demand_final = demand_final[
    [
        "datetime",
        "date",
        "hour",
        "day_of_week",
        "day_of_week_num",
        "day_of_month",
        "month",
        "is_weekend",
        "PULocationID",
        "Borough",
        "Zone",
        "service_zone",
        "demand"
    ]
]

demand_final.head(10)

,datetime,date,hour,day_of_week,day_of_week_num,day_of_month,month,is_weekend,PULocationID,Borough,Zone,service_zone,demand
0,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,1,EWR,Newark Airport,EWR,0
1,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,2,Queens,Jamaica Bay,Boro Zone,0
2,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,3,Bronx,Allerton/Pelham Gardens,Boro Zone,74
3,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,4,Manhattan,Alphabet City,Yellow Zone,242
4,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,5,Staten Island,Arden Heights,Boro Zone,18
5,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,15
6,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,7,Queens,Astoria,Boro Zone,616
7,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,8,Queens,Astoria Park,Boro Zone,5
8,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,9,Queens,Auburndale,Boro Zone,31
9,2025-01-01,2025-01-01,0,Wednesday,2,1,1,False,10,Queens,Baisley Park,Boro Zone,134


In [26]:
print("Shape:", demand_final.shape)

print("\nMissing values:")
print(demand_final.isnull().sum())

print("\nDemand total:")
print(f"{demand_final['demand'].sum():,}")

print("\nZero-demand rows:")
print((demand_final["demand"] == 0).sum())

print("\nDate range:")
print(demand_final["datetime"].min())
print(demand_final["datetime"].max())

print("\nUnique zones:")
print(demand_final["PULocationID"].nunique())

Shape: (195672, 13)

Missing values:
datetime           0
date               0
hour               0
day_of_week        0
day_of_week_num    0
day_of_month       0
month              0
is_weekend         0
PULocationID       0
Borough            0
Zone               0
service_zone       0
demand             0
dtype: int64

Demand total:
20,404,893

Zero-demand rows:
9112

Date range:
2025-01-01 00:00:00
2025-01-31 23:00:00

Unique zones:
263


In [27]:
demand_final.to_parquet(
    "../data/processed/hourly_zone_demand_2025_01.parquet",
    index=False
)

demand_final.to_csv(
    "../data/processed/hourly_zone_demand_2025_01.csv",
    index=False
)

print("Demand datasets saved successfully.")

Demand datasets saved successfully.
